The notebook is part of the original Residual_Plot.ipynb concentrating in loading and calculating disk object.

1. Add Auto-Reload so that changes in the disk class Python script will be auto-updated

In [1]:
# Autoreload the imported modules
%load_ext autoreload
%autoreload 2


In [2]:
# import necessary libraries

import os  
import sys
import numpy as np  
import re  # Python’s regular expressions module to extract numbers from filenames
from astropy.io import fits
import pandas as pd
from gofish import imagecube
import matplotlib.pyplot as plt
import glob   # to find files matching a pattern
from reproject import reproject_interp
from IPython.display import display
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
 
import pickle # to save and load Python objects

# 
# from disk_residuals_median_SNR import DiskResiduals_Median_SNR as DiskResiduals


sys.path.insert(0, os.path.abspath('.'))

import disk_residuals_median_SNR
import importlib
importlib.reload(disk_residuals_median_SNR)
DiskResiduals = disk_residuals_median_SNR.DiskResiduals_Median_SNR


Load comprehensive disk datasets including clean and residual images for each protoplanetary disk, along with published geometrical parameters, radial brightness profiles, and gap/ring structural information across different robust weighting values.

Upon loading the specified file paths, store all datasets in the all_disks dictionary for systematic analysis. The hierarchical data structure will be documented in the accompanying tree diagram.

In [3]:
# =============================================================================
# DATA LOADING AND INITIALIZATION
# =============================================================================

# Define data directories
data_dir = "D:/exoALMA_disk_data"  # Main data directory
home_dir = "D:/CPD_MPIA/"          # Project home directory

# Load disk distances from reference file
dist_df = pd.read_csv(home_dir + "disk_distances.txt", 
                      comment="#", 
                      names=["disk_name", "distance_pc"])
distance_dict = dict(zip(dist_df["disk_name"], dist_df["distance_pc"]))

# Initialize main data structure
all_disks = {}

# =============================================================================
# MAIN DISK LOADING LOOP
# =============================================================================

for disk in os.listdir(data_dir):
    
    # -------------------------------------------------------------------------
    # Define file and directory paths for current disk
    # -------------------------------------------------------------------------
    res_path = os.path.join(data_dir, disk, "images_frank_residuals_different_robust")
    clean_path = os.path.join(data_dir, disk, "images_data_different_robust")
    geom_file = os.path.join(data_dir, disk, f"{disk}_geometrical_parameters_continuum_galario.txt")
    clean_profile_file = os.path.join(data_dir, disk, f"{disk}_CLEAN_profile_robust-05.txt")
    radii_file = os.path.join(data_dir, disk, f"{disk}_continuum_radii.txt")
    ringgap_file = os.path.join(data_dir, disk, f"{disk}_gaps_rings_continuum.txt")
    
    # -------------------------------------------------------------------------
    # Check if essential files exist before processing
    # -------------------------------------------------------------------------
    if os.path.isdir(res_path) and os.path.exists(geom_file):
        
        # ---------------------------------------------------------------------
        # Initialize disk object and load core data
        # ---------------------------------------------------------------------
        disk_obj = DiskResiduals(disk, res_path, geom_file)
        disk_obj.load_residuals()
        
        # ---------------------------------------------------------------------
        # Load optional datasets if available
        # ---------------------------------------------------------------------
        # Clean images
        if os.path.isdir(clean_path):
            disk_obj.load_clean_images(clean_path)
        
        # Radial profile
        if os.path.exists(clean_profile_file):
            disk_obj.load_clean_profile(clean_profile_file)
        
        # Disk size information
        if os.path.exists(radii_file):
            disk_obj.load_disksize(radii_file)
        
        # Gap and ring structural information
        if os.path.exists(ringgap_file):
            disk_obj.load_ringgap(ringgap_file)
        
        # ---------------------------------------------------------------------
        # Add distance information from lookup table
        # ---------------------------------------------------------------------
        disk_obj.distance_pc = distance_dict.get(disk, None)
        
        # ---------------------------------------------------------------------
        # Extract beam information for all robust weightings
        # ---------------------------------------------------------------------
        disk_obj.beam_info = {}  # Initialize beam info dictionary
        
        for robust_val in ["-2.0", "-1.5", "-1.0", "-0.5", "0.0", "0.5", "1.0", "1.5", "2.0"]:
            fits_path_resid = os.path.join(res_path, f"{disk}_continuum_resid_robust{robust_val}.image.fits")
            
            if os.path.exists(fits_path_resid):
                with fits.open(fits_path_resid) as hdul:
                    hdr = hdul[0].header
                    
                    # Extract beam and pixel information
                    beam_x = float(hdr.get('BMAJ', 0)) * 3600.0      # Major axis (arcsec)
                    beam_y = float(hdr.get('BMIN', 0)) * 3600.0      # Minor axis (arcsec)  
                    pix_scale = abs(float(hdr.get('CDELT1', 0))) * 3600.0  # Pixel scale (arcsec)
                    beam_area_pix = (beam_x * beam_y) / (pix_scale ** 2)   # Beam area (pixels)
                    
                    # Store beam information for this robust value
                    disk_obj.beam_info[robust_val] = {
                        "beam_x_arcsec": beam_x,
                        "beam_y_arcsec": beam_y,
                        "pixel_scale_arcsec": pix_scale,
                        "beam_area_pix": beam_area_pix
                    }
        
        # ---------------------------------------------------------------------
        # Add completed disk object to main dictionary
        # ---------------------------------------------------------------------
        all_disks[disk] = disk_obj

# =============================================================================
# SUMMARY OUTPUT
# =============================================================================
print(f"Successfully loaded {len(all_disks)} disk objects into all_disks dictionary.")
print(f"Each DiskResiduals object contains residuals, geometries, and observational metadata.")
print(f"DiskResiduals is a class that contains {len(all_disks)} disks with their residuals loaded. \nall_disks is a dictionary with disk names as keys and DiskResiduals objects as values.\ndisk_obj is an object of DiskResiduals class that contains the residuals and geometries for each disk.")

[WARN] PDS_66: D:/exoALMA_disk_data\PDS_66\PDS_66_gaps_rings_continuum.txt is empty or only comments.
Successfully loaded 15 disk objects into all_disks dictionary.
Each DiskResiduals object contains residuals, geometries, and observational metadata.
DiskResiduals is a class that contains 15 disks with their residuals loaded. 
all_disks is a dictionary with disk names as keys and DiskResiduals objects as values.
disk_obj is an object of DiskResiduals class that contains the residuals and geometries for each disk.
Successfully loaded 15 disk objects into all_disks dictionary.
Each DiskResiduals object contains residuals, geometries, and observational metadata.
DiskResiduals is a class that contains 15 disks with their residuals loaded. 
all_disks is a dictionary with disk names as keys and DiskResiduals objects as values.
disk_obj is an object of DiskResiduals class that contains the residuals and geometries for each disk.


Remove trailing dots from residuals dictionary keys to ensure consistency with sigma_masks naming convention. This standardization is necessary for proper key matching in downstream analysis.

In [4]:

# =============================================================================
# DATA POST-PROCESSING: CLEAN RESIDUALS KEYS
# =============================================================================

# Remove trailing dots from residuals keys to match sigma_masks convention
for disk_name, disk_obj in all_disks.items():
    # Create a copy of the original residuals dictionary
    old_residuals = dict(disk_obj.residuals)
    
    # Clear the original residuals dictionary
    disk_obj.residuals.clear()
    
    # Rebuild dictionary with cleaned keys (no trailing dots)
    for key, value in old_residuals.items():
        clean_key = key.rstrip('.')           # Remove trailing dot(s)
        disk_obj.residuals[clean_key] = value # Assign cleaned key with its value

Gap Analysis: Extract and Display Gap Information

Extract gap locations (flag=0) from all disk objects for systematic analysis and future injection-recovery testing. This creates a consolidated DataFrame containing only gap features across all disks, facilitating statistical analysis and parameter manipulation for synthetic source injection studies.

In [5]:
# =============================================================================
# GAP ANALYSIS: EXTRACT GAP INFORMATION FOR ALL DISKS
# =============================================================================

# Initialize list to collect gap data from all disks
gap_data = []

# Extract gap information from each disk object
for disk_name, disk_obj in all_disks.items():
    # Check if disk has gap/ring information loaded
    if hasattr(disk_obj, 'ringgap_info') and "flag" in disk_obj.ringgap_info:
        # Filter for gaps only (flag == 0, rings have flag == 1)
        gaps = disk_obj.ringgap_info["flag"] == 0
        
        if gaps.any():
            # Get all property names except 'flag'
            labels = [k for k in disk_obj.ringgap_info.keys() if k != "flag"]
            arrays = [disk_obj.ringgap_info[label][gaps] for label in labels]
            
            # Create row for each gap
            for values in zip(*arrays):
                row = {'disk_name': disk_name}
                row.update({label: val for label, val in zip(labels, values)})
                gap_data.append(row)

# Create consolidated gaps DataFrame
gap_df = pd.DataFrame(gap_data)

# Display gap summary
print(f"Found {len(gap_df)} gaps across {gap_df['disk_name'].nunique()} disks")
display(gap_df)

Found 27 gaps across 12 disks


,disk_name,radius_au,radius_arcsec,width_au,width_arcsec,gap_depth,r_in_au,r_in_arcsec,r_out_au,r_out_arcsec
0,AA_Tau,11.0,0.082,28.1,0.209,0.01,4.9,0.037,33.0,0.245
1,AA_Tau,64.3,0.478,8.2,0.061,0.44,60.3,0.448,68.5,0.508
2,AA_Tau,79.8,0.593,10.2,0.076,0.34,75.3,0.559,85.5,0.635
3,AA_Tau,105.3,0.782,4.9,0.036,0.94,103.1,0.766,108.0,0.802
4,DM_Tau,13.5,0.094,12.7,0.088,0.08,7.7,0.053,20.4,0.142
5,DM_Tau,71.8,0.498,18.7,0.130,0.78,64.4,0.447,83.1,0.577
6,DM_Tau,102.5,0.712,6.6,0.046,0.92,99.2,0.689,105.8,0.735
7,HD_135344B,13.2,0.098,40.8,0.302,0.00,1.5,0.011,42.3,0.313
8,HD_135344B,66.5,0.493,11.4,0.084,0.47,60.9,0.451,72.3,0.535
9,HD_143006,21.8,0.132,18.3,0.111,0.10,13.8,0.084,32.1,0.195


Image Reprojection: Align Clean Images with Residuals

Reproject full field-of-view clean images to match the coordinate system and pixel grid of the corresponding residual images. This ensures spatial consistency between clean and residual data for accurate analysis and comparison. Only processes robust=2.0 images and skips existing reprojected files to avoid redundant processing.

In [6]:
# =============================================================================
# IMAGE REPROJECTION: COORDINATE ALIGNMENT
# =============================================================================

# Configuration for reprojection
data_dir = "D:/exoALMA_disk_data"
robust_val = "2.0"
valid_disks = list(all_disks.keys())

for disk_name in os.listdir(data_dir):
    base_path = os.path.join(data_dir, disk_name)
    
    # -------------------------------------------------------------------------
    # Validate directory and disk selection
    # -------------------------------------------------------------------------
    if not os.path.isdir(base_path):
        continue
    
    if disk_name not in valid_disks:
        print(f"Skipping non-disk folder: {disk_name}")
        continue
    
    # -------------------------------------------------------------------------
    # Define file paths for reprojection
    # -------------------------------------------------------------------------
    fits_path_clean_fullfov = os.path.join(
        base_path, "images_data_different_robust",
        f"{disk_name}_continuum_data_robust{robust_val}_FullFOV.image.fits"
    )
    fits_path_resid = os.path.join(
        base_path, "images_frank_residuals_different_robust",
        f"{disk_name}_continuum_resid_robust{robust_val}.image.fits"
    )
    fits_path_clean_out = os.path.join(
        base_path, "images_data_different_robust",
        f"{disk_name}_continuum_data_robust{robust_val}.image.fits"
    )
    
    # -------------------------------------------------------------------------
    # Check if reprojection already completed
    # -------------------------------------------------------------------------
    if os.path.exists(fits_path_clean_out):
        print(f"Skipping reprojection for {disk_name} (robust {robust_val}): output file already exists.")
        continue
    
    # -------------------------------------------------------------------------
    # Perform coordinate reprojection
    # -------------------------------------------------------------------------
    try:
        # Load source and target images
        clean_hdu = fits.open(fits_path_clean_fullfov)[0]
        resid_hdu = fits.open(fits_path_resid)[0]
        
        # Reproject clean image to match residual coordinates
        reproj_data, _ = reproject_interp(clean_hdu, resid_hdu.header)
        
        # Save reprojected image
        fits.writeto(fits_path_clean_out, reproj_data, resid_hdu.header, overwrite=False)
        print(f"✓ {disk_name} reprojected successfully")
        
    except Exception as e:
        print(f"✗ Error reprojecting {disk_name}: {e}")

print("\nReprojection process completed.")

Skipping reprojection for AA_Tau (robust 2.0): output file already exists.
Skipping reprojection for CQ_Tau (robust 2.0): output file already exists.
Skipping reprojection for DM_Tau (robust 2.0): output file already exists.
Skipping reprojection for HD_135344B (robust 2.0): output file already exists.
Skipping reprojection for HD_143006 (robust 2.0): output file already exists.
Skipping reprojection for HD_34282 (robust 2.0): output file already exists.
Skipping reprojection for J1604 (robust 2.0): output file already exists.
Skipping reprojection for J1615 (robust 2.0): output file already exists.
Skipping reprojection for J1842 (robust 2.0): output file already exists.
Skipping reprojection for J1852 (robust 2.0): output file already exists.
Skipping reprojection for LkCa_15 (robust 2.0): output file already exists.
Skipping reprojection for MWC_758 (robust 2.0): output file already exists.
Skipping reprojection for PDS_66 (robust 2.0): output file already exists.
Skipping reproject

## CPD Emission Model Data Loading

Load circumplanetary disk (CPD) emission model parameters, detection results, and analysis data from the Zhu model framework. This includes disk properties, five-sigma source detections, planet kink locations, and model grid results for injection-recovery testing.

In [7]:
# =============================================================================
# CPD DISK PROPERTIES (from Zhu model framework)
# =============================================================================

# Extended disk properties for CPD modeling (ENHANCED)
# Format: [disk_name, M_star (Msun), distance (pc), L_star (Lsun), beam (arcsec), inclination (deg), RA, Dec]
disk_arr = {
    "AA_Tau": ["AA_Tau", 0.79, 145, 1.1, 0.12, 58.54, "04 34 55.420", "+24 28 53.034"],
    "CQ_Tau": ["CQ_Tau", 1.4, 149, 10, 0.135, 35.24, "05 35 58.467", "+24 44 54.091"],
    "DM_Tau": ["DM_Tau", 0.45, 144, 0.24, 0.134, 35.97, "04 33 48.733", "+18 10 09.973"],
    "HD_34282": ["HD_34282", 1.61, 309, 108, 0.106, 59.09, "05 16 00.477", "-09 48 35.394"],
    "HD_135344B": ["HD_135344B", 1.61, 135, 6.7, 0.26, 20.73, "15 15 48.446", "-37 09 16.024"],
    "HD_143006": ["HD_143006", 1.56, 167, 3.8, 0.226, 18.69, "15 58 36.913", "-22 57 15.221"],
    "J1604": ["J1604", 1.29, 112, 0.76, 0.257, 8.72, "16 04 21.642", "-21 30 29.058"],
    "J1615": ["J1615", 1.14, 156, 1.07, 0.303, 47.10, "16 15 20.234", "-32 55 05.099"],
    "J1842": ["J1842", 1.07, 151, 0.8, 0.235, 39.22, "18 42 57.981", "-35 32 42.827"],
    "J1852": ["J1852", 1.03, 147, 0.6, 0.236, 32.50, "18 52 17.301", "-37 00 11.949"],
    "LkCa_15": ["LkCa_15", 1.14, 156, 1, 0.131, 50.59, "04 39 17.791", "+22 21 03.390"],
    "MWC_758": ["MWC_758", 1.4, 151, 10.4, 0.241, 7.27, "05 30 27.529", "+25 19 57.076"],
    "PDS_66": ["PDS_66", 1.28, 98, 1.2, 0.234, 32.02, "13 22 07.542", "-69 38 12.219"],
    "SY_Cha": ["SY_Cha", 0.77, 182, 0.55, 0.178, 51.65, "10 56 30.388", "-77 11 39.402"],
    "V4046_Sgr": ["V4046_Sgr", 1.73, 72, 0.5, 0.241, 33.36, "18 14 10.482", "-32 47 34.517"]
}

print("CPD disk properties loaded successfully.")
print(f"Properties available for {len(disk_arr)} disks.")

CPD disk properties loaded successfully.
Properties available for 15 disks.


In [8]:
# =============================================================================
# FIVE SIGMA SOURCE DETECTIONS
# =============================================================================

# Five sigma source detections from CPD analysis (ENHANCED)
# Format: [disk_name, radius_au, flux_ujy, sigma_flux_ujy, PB_correction, detection_sigma]
five_sigma_sources = [
    ["CQ_Tau", 165.37, 260.30, 45.81, 0.99, 5.08],
    ["CQ_Tau", 932.14, 1455, 40.25, 0.76, 5.46],
    ["DM_Tau", 1845.47, 5422.67, 161.82, 0.36, 9.46],
    ["V4046_Sgr", 837.63, 90.54, 22.22, 0.42, 5.24],
]

# Planet kink sources with sigma flux
# Format: [disk_name, radius_au, sigma_flux_ujy]
planet_kink_sources = [
    ["AA_Tau", 80, 136.85],
    ["SY_Cha", 140, 57.23],
    ["J1842", 105, 35.29],
    ["J1615", 310, 24.56],
    ["LkCa_15", 240, 20.14],
    ["HD_143006", 32, 896.96],
]



In [9]:
# =============================================================================
# APPLY CPD DATA TO DISK OBJECTS
# =============================================================================

# Apply CPD-specific properties to each disk object
for disk_name, disk_obj in all_disks.items():
    # -------------------------------------------------------------------------
    # CPD disk properties
    # -------------------------------------------------------------------------
    if disk_name in disk_arr:
        disk_obj.stellar_mass_zhu = disk_arr[disk_name][1]  # M_sun
        disk_obj.stellar_luminosity = disk_arr[disk_name][3]  # L_sun
        disk_obj.beam_arcsec = disk_arr[disk_name][4]  # arcsec
        disk_obj.inclination_deg = disk_arr[disk_name][5]  # degrees
        disk_obj.ra_str = disk_arr[disk_name][6]  # RA string
        disk_obj.dec_str = disk_arr[disk_name][7]  # Dec string
    
    # -------------------------------------------------------------------------
    # Five sigma detections for this disk
    # -------------------------------------------------------------------------
    disk_detections = [det for det in five_sigma_sources if det[0] == disk_name]
    disk_obj.five_sigma_detections = disk_detections
    
    # -------------------------------------------------------------------------
    # Planet kink detections for this disk
    # -------------------------------------------------------------------------
    disk_kinks = [kink for kink in planet_kink_sources if kink[0] == disk_name]
    disk_obj.planet_kink_detections = disk_kinks




In [10]:
# =============================================================================
# SAVE FILES TO SUBFOLDER
# =============================================================================

import json


# Create output directory
output_dir = "disk_object_files"
os.makedirs(output_dir, exist_ok=True)

# Save main data files
with open(os.path.join(output_dir, "all_disks.pkl"), 'wb') as f:
    pickle.dump(all_disks, f)

gap_df.to_csv(os.path.join(output_dir, "gaps.csv"), index=False)
gap_df.to_pickle(os.path.join(output_dir, "gaps.pkl"))

with open(os.path.join(output_dir, "disk_properties.json"), 'w') as f:
    json.dump(disk_arr, f, indent=2)

# Save detection sources as JSON files
with open(os.path.join(output_dir, "five_sigma_sources.json"), 'w') as f:
    json.dump(five_sigma_sources, f, indent=2)

with open(os.path.join(output_dir, "planet_kink_sources.json"), 'w') as f:
    json.dump(planet_kink_sources, f, indent=2)

print(f"Files saved to: {output_dir}/")
print(f"  ├── all_disks.pkl")
print(f"  ├── gaps.csv")
print(f"  ├── gaps.pkl")
print(f"  ├── disk_properties.json")
print(f"  ├── five_sigma_sources.json")
print(f"  └── planet_kink_sources.json")

Files saved to: disk_object_files/
  ├── all_disks.pkl
  ├── gaps.csv
  ├── gaps.pkl
  ├── disk_properties.json
  ├── five_sigma_sources.json
  └── planet_kink_sources.json


In [11]:
# =============================================================================
# DISK OBJECT STRUCTURE ANALYSIS
# =============================================================================

def print_disk_object_structure(disk_name=None, detailed=False):
    """
    Print the structure and attributes of disk objects.
    
    Parameters:
    -----------
    disk_name : str, optional  
        Specific disk to analyze. If None, shows summary for all disks.
    detailed : bool
        If True, shows detailed information including array shapes and sample values.
    """
    
    if disk_name:
        # Analyze specific disk
        if disk_name not in all_disks:
            print(f"❌ Disk '{disk_name}' not found in all_disks dictionary")
            print(f"Available disks: {list(all_disks.keys())}")
            return
        
        disk_obj = all_disks[disk_name]
        print(f"🔍 DETAILED STRUCTURE FOR: {disk_name}")
        print("=" * 60)
        
    else:
        # Show summary for all disks
        print(f"📊 DISK OBJECT COLLECTION SUMMARY")
        print("=" * 60)
        print(f"Total disks loaded: {len(all_disks)}")
        print(f"Disk names: {list(all_disks.keys())}")
        print()
        
        # Use first disk as example
        disk_name = list(all_disks.keys())[0]
        disk_obj = all_disks[disk_name]
        print(f"📋 STRUCTURE EXAMPLE (using {disk_name}):")
        print("-" * 40)
    
    # Analyze disk object attributes
    attributes = []
    for attr_name in dir(disk_obj):
        if not attr_name.startswith('_'):  # Skip private attributes
            attr_value = getattr(disk_obj, attr_name)
            if not callable(attr_value):  # Skip methods
                attributes.append((attr_name, attr_value))
    
    # Categorize attributes
    basic_info = []
    arrays_dicts = []
    detection_info = []
    cpd_properties = []
    
    for attr_name, attr_value in attributes:
        if attr_name in ['disk_name', 'distance_pc']:
            basic_info.append((attr_name, attr_value))
        elif attr_name in ['five_sigma_detections', 'planet_kink_detections']:
            detection_info.append((attr_name, attr_value))
        elif attr_name in ['stellar_mass_zhu', 'stellar_luminosity', 'beam_arcsec', 
                          'inclination_deg', 'ra_str', 'dec_str']:
            cpd_properties.append((attr_name, attr_value))
        elif isinstance(attr_value, (dict, list, np.ndarray)):
            arrays_dicts.append((attr_name, attr_value))
        else:
            basic_info.append((attr_name, attr_value))
    
    # Print categorized information
    print("\n🏷️  BASIC INFORMATION:")
    for attr_name, attr_value in basic_info:
        if isinstance(attr_value, (int, float, str)) or attr_value is None:
            print(f"   {attr_name}: {attr_value}")
        else:
            print(f"   {attr_name}: {type(attr_value).__name__}")
    
    print("\n🌟 CPD PROPERTIES:")
    for attr_name, attr_value in cpd_properties:
        print(f"   {attr_name}: {attr_value}")
    
    print("\n🎯 DETECTION DATA:")
    for attr_name, attr_value in detection_info:
        if isinstance(attr_value, list):
            print(f"   {attr_name}: {len(attr_value)} detections")
            if detailed and len(attr_value) > 0:
                for i, det in enumerate(attr_value):
                    print(f"     └─ Detection {i+1}: {det}")
        else:
            print(f"   {attr_name}: {attr_value}")
    
    print("\n📊 ARRAYS & DICTIONARIES:")
    for attr_name, attr_value in arrays_dicts:
        if isinstance(attr_value, dict):
            print(f"   {attr_name}: dict with {len(attr_value)} keys")
            if detailed:
                for key in list(attr_value.keys())[:3]:  # Show first 3 keys
                    val = attr_value[key]
                    if isinstance(val, np.ndarray):
                        print(f"     └─ {key}: array shape {val.shape}")
                    elif isinstance(val, dict):
                        print(f"     └─ {key}: dict with {len(val)} keys") 
                    else:
                        print(f"     └─ {key}: {type(val).__name__}")
                if len(attr_value) > 3:
                    print(f"     └─ ... and {len(attr_value)-3} more keys")
                    
        elif isinstance(attr_value, list):
            print(f"   {attr_name}: list with {len(attr_value)} items")
            if detailed and len(attr_value) > 0:
                print(f"     └─ Sample item: {attr_value[0]}")
                
        elif isinstance(attr_value, np.ndarray):
            print(f"   {attr_name}: array shape {attr_value.shape}, dtype {attr_value.dtype}")
            if detailed:
                print(f"     └─ Sample values: {attr_value.flat[:3] if attr_value.size > 0 else 'empty'}")
        else:
            print(f"   {attr_name}: {type(attr_value).__name__}")

# Show overall structure
print_disk_object_structure()

print("\n" + "="*80)
print("💡 USAGE EXAMPLES:")
print("   # Show detailed structure for specific disk:")
print("   print_disk_object_structure('AA_Tau', detailed=True)")
print("")  
print("   # Access specific disk data:")
print("   disk = all_disks['AA_Tau']")
print("   print(f'Distance: {disk.distance_pc} pc')")
print("   print(f'Stellar mass: {disk.stellar_mass_zhu} Msun')")
print("   print(f'Five-sigma detections: {len(disk.five_sigma_detections)}')")

📊 DISK OBJECT COLLECTION SUMMARY
Total disks loaded: 15
Disk names: ['AA_Tau', 'CQ_Tau', 'DM_Tau', 'HD_135344B', 'HD_143006', 'HD_34282', 'J1604', 'J1615', 'J1842', 'J1852', 'LkCa_15', 'MWC_758', 'PDS_66', 'SY_Cha', 'V4046_Sgr']

📋 STRUCTURE EXAMPLE (using AA_Tau):
----------------------------------------

🏷️  BASIC INFORMATION:
   PA: 93.77079777
   center: tuple
   distance_pc: 135
   inc: 58.53531224
   name: AA_Tau
   path: D:/exoALMA_disk_data\AA_Tau\images_frank_residuals_different_robust
   rms_noise_FullFOV: None

🌟 CPD PROPERTIES:
   beam_arcsec: 0.12
   dec_str: +24 28 53.034
   inclination_deg: 58.54
   ra_str: 04 34 55.420
   stellar_luminosity: 1.1
   stellar_mass_zhu: 0.79

🎯 DETECTION DATA:
   five_sigma_detections: 0 detections
   planet_kink_detections: 1 detections

📊 ARRAYS & DICTIONARIES:
   beam_info: dict with 8 keys
   clean_images: dict with 10 keys
   clean_profile: dict with 3 keys
   disksize: dict with 6 keys
   residuals: dict with 8 keys
   ringgap: array 